In [1]:
# from pathlib import Path
# import os
# import torch

# print("Python environment restarted successfully")
# print("PyTorch:", torch.__version__)
# print("CUDA available:", torch.cuda.is_available())

# root = Path("/kaggle/input")

# print("\nInput directory exists:", root.exists())

# if root.exists():
#     print("Attached dataset directories:")
#     for p in sorted(root.iterdir()):
#         print(" -", p.name)

# print("\nChecking expected dataset root:")
# expected = root / "datasets" / "saisdsaiiitbh"

# print("Exists:", expected.exists())

# if expected.exists():
#     for p in sorted(expected.iterdir()):
#         print(p.name, "| directory:", p.is_dir())

Python environment restarted successfully
PyTorch: 2.11.0+cu128
CUDA available: True

Input directory exists: True
Attached dataset directories:
 - datasets

Checking expected dataset root:
Exists: True
dff-code | directory: True
eval-tar | directory: True
ff-tar | directory: True


In [2]:
from pathlib import Path

BASE = Path("/kaggle/input/datasets/saisdsaiiitbh")

CODE_ROOT = BASE / "dff-code"
FF_ROOT = BASE / "ff-tar"
EVAL_ROOT = BASE / "eval-tar"

for name, root in [
    ("Code", CODE_ROOT),
    ("FF++ crops", FF_ROOT),
    ("Evaluation crops", EVAL_ROOT),
]:
    print(f"\n{'=' * 20} {name} {'=' * 20}")
    print("Path:", root)
    print("Exists:", root.exists())

    if root.exists():
        for item in sorted(root.iterdir())[:20]:
            kind = "DIR " if item.is_dir() else "FILE"
            print(f"  [{kind}] {item.name}")

# Check whether the Python source files are available.
print("\nChecking required source files:")

required = [
    "dff_adapter.py",
    "dataset.py",
    "train.py",
    "evaluate_dff.py",
    "test_dff_adapter.py",
]

for filename in required:
    matches = list(CODE_ROOT.rglob(filename))
    print(f"{filename}: {matches[0] if matches else 'NOT FOUND'}")


==================== Code ====================
Path: /kaggle/input/datasets/saisdsaiiitbh/dff-code
Exists: True
  [FILE] bench.py
  [FILE] dataset.py
  [FILE] dff_adapter.py
  [FILE] evaluate_dff.py
  [FILE] run_experiments.py
  [FILE] test_dff_adapter.py
  [FILE] train.py

==================== FF++ crops ====================
Path: /kaggle/input/datasets/saisdsaiiitbh/ff-tar
Exists: True
  [DIR ] ffpp
  [FILE] ffpp_manifest_split.csv

==================== Evaluation crops ====================
Path: /kaggle/input/datasets/saisdsaiiitbh/eval-tar
Exists: True
  [DIR ] cdf1
  [FILE] cdf1_manifest.csv
  [DIR ] cdf2
  [FILE] cdf2_manifest.csv

Checking required source files:
dff_adapter.py: /kaggle/input/datasets/saisdsaiiitbh/dff-code/dff_adapter.py
dataset.py: /kaggle/input/datasets/saisdsaiiitbh/dff-code/dataset.py
train.py: /kaggle/input/datasets/saisdsaiiitbh/dff-code/train.py
evaluate_dff.py: /kaggle/input/datasets/saisdsaiiitbh/dff-code/evaluate_dff.py
test_dff_adapter.py: /kaggle/in

In [4]:
import os

search_roots = [FF_ROOT, EVAL_ROOT]
target_names = {
    "ffpp_manifest_split.csv",
    "cdf1_manifest.csv",
    "cdf2_manifest.csv",
}

found = {}

for root in search_roots:
    if not root.exists():
        continue

    for current, dirs, files in os.walk(root):
        for filename in target_names.intersection(files):
            path = Path(current) / filename
            found.setdefault(filename, path)

        if target_names.issubset(found):
            break

print("Manifest search results:")

for filename in sorted(target_names):
    print(f"{filename}: {found.get(filename, 'NOT FOUND')}")

Manifest search results:
cdf1_manifest.csv: /kaggle/input/datasets/saisdsaiiitbh/eval-tar/cdf1_manifest.csv
cdf2_manifest.csv: /kaggle/input/datasets/saisdsaiiitbh/eval-tar/cdf2_manifest.csv
ffpp_manifest_split.csv: /kaggle/input/datasets/saisdsaiiitbh/ff-tar/ffpp_manifest_split.csv


In [5]:
import subprocess
import sys

CODE_DIR = next(
    p.parent
    for p in CODE_ROOT.rglob("train.py")
    if all((p.parent / f).exists() for f in [
        "dff_adapter.py",
        "dataset.py",
        "train.py",
        "evaluate_dff.py",
        "test_dff_adapter.py",
    ])
)

print("Using code directory:", CODE_DIR)

result = subprocess.run(
    [sys.executable, "-u", "test_dff_adapter.py"],
    cwd=CODE_DIR,
    text=True,
)

if result.returncode != 0:
    raise RuntimeError("Model tests failed. Do not start training yet.")

print("Model tests passed.")

Using code directory: /kaggle/input/datasets/saisdsaiiitbh/dff-code
PASS  test_adapters_wrap_q_v_dense_in_every_block
PASS  test_each_input_chunk_used_exactly_once
PASS  test_gradient_flow_and_unselected_logits
PASS  test_linear_probe_variant
PASS  test_lora_baseline_variant
PASS  test_matches_naive_equations
PASS  test_only_adapters_and_heads_train
      Small  adapter params =    665,064
      Base   adapter params =  1,328,616
      Large  adapter params =  3,541,968
PASS  test_parameter_budget_matches_paper
      never-selected (adapter,head,expert) slots: 28/144 = 19.4%
PASS  test_routing_report_runs
PASS  test_task_switch_changes_output
PASS  test_top3_routing_and_normalisation
PASS  test_zero_init_is_identity

12/12 passed
Model tests passed.


In [6]:
import sys
import torch
from pathlib import Path

CODE_DIR = Path("/kaggle/input/datasets/saisdsaiiitbh/dff-code")
FF_ROOT = Path("/kaggle/input/datasets/saisdsaiiitbh/ff-tar")
FF_MANIFEST = FF_ROOT / "ffpp_manifest_split.csv"

EVAL_ROOT = Path("/kaggle/input/datasets/saisdsaiiitbh/eval-tar")
CDF1_MANIFEST = EVAL_ROOT / "cdf1_manifest.csv"
CDF2_MANIFEST = EVAL_ROOT / "cdf2_manifest.csv"

sys.path.insert(0, str(CODE_DIR))

from dataset import FrameDataset

print("GPU:", torch.cuda.get_device_name(0))
print("FF++ manifest exists:", FF_MANIFEST.exists())
print("CDF-v1 manifest exists:", CDF1_MANIFEST.exists())
print("CDF-v2 manifest exists:", CDF2_MANIFEST.exists())

# Construct the dataset index using the existing implementation.
# This may take a little time because it enumerates crop files.
train_ds = FrameDataset(
    crops_root=str(FF_ROOT),
    manifest=str(FF_MANIFEST),
    split="train",
    mode="eval",
    frames_per_video=1,
)

print("\nTraining videos:", len(train_ds.video_labels))
print("Indexed samples:", len(train_ds))

# Read and preprocess one actual crop.
image, authenticity, forgery_type, video_idx = train_ds[0]

print("\nActual crop loaded successfully.")
print("Image shape:", tuple(image.shape))
print("Image dtype:", image.dtype)
print("Authenticity label:", authenticity.item())
print("Forgery-type label:", forgery_type.item())
print("Video index:", video_idx.item())

assert image.shape == (3, 224, 224)
assert torch.isfinite(image).all()
assert authenticity.item() in (0.0, 1.0)
assert 0 <= forgery_type.item() < 5

print("\nReal-data loading and preprocessing passed.")

GPU: Tesla T4
FF++ manifest exists: True
CDF-v1 manifest exists: True
CDF-v2 manifest exists: True

Training videos: 3600
Indexed samples: 3600

Actual crop loaded successfully.
Image shape: (3, 224, 224)
Image dtype: torch.float32
Authenticity label: 1.0
Forgery-type label: 1
Video index: 0

Real-data loading and preprocessing passed.


In [7]:
import sys
import torch
from torch.utils.data import DataLoader

CODE_DIR = "/kaggle/input/datasets/saisdsaiiitbh/dff-code"
sys.path.insert(0, CODE_DIR)

from dataset import FrameDataset
from train import build_model

device = torch.device("cuda")

# Load two real FF++ training samples.
ds = FrameDataset(
    crops_root=str(FF_ROOT),
    manifest=str(FF_MANIFEST),
    split="train",
    mode="eval",
    frames_per_video=1,
)

loader = DataLoader(ds, batch_size=2, shuffle=False, num_workers=0)
images, auth_labels, type_labels, video_indices = next(iter(loader))

images = images.to(device)

print("Loading DINOv2-Small + DFF-Adapter...")
model = build_model("small", variant="dff").to(device)
model.eval()

print("Model loaded. Running forward pass...")

with torch.inference_mode():
    with torch.autocast(device_type="cuda", dtype=torch.float16):
        auth_logits = model(images, task=0)
        type_logits = model(images, task=1)

auth_logits = auth_logits.float()
type_logits = type_logits.float()

print("\nForward pass successful.")
print("GPU:", torch.cuda.get_device_name(0))
print("Batch shape:", tuple(images.shape))
print("Authenticity logits shape:", tuple(auth_logits.shape))
print("Forgery-type logits shape:", tuple(type_logits.shape))
print("Authenticity logits:", auth_logits.cpu().tolist())
print("Forgery-type predictions:", type_logits.argmax(dim=1).cpu().tolist())

assert auth_logits.shape[0] == 2
assert type_logits.shape == (2, 5)
assert torch.isfinite(auth_logits).all()
assert torch.isfinite(type_logits).all()

del model, images, auth_logits, type_logits, loader, ds
torch.cuda.empty_cache()

print("\nPASS: Real crops + DINOv2-Small + DFF-Adapter work on the T4.")

Loading DINOv2-Small + DFF-Adapter...


config.json:   0%|          | 0.00/547 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 88.2MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/223 [00:00<?, ?it/s]

Model loaded. Running forward pass...

Forward pass successful.
GPU: Tesla T4
Batch shape: (2, 3, 224, 224)
Authenticity logits shape: (2,)
Forgery-type logits shape: (2, 5)
Authenticity logits: [0.73828125, 1.4111328125]
Forgery-type predictions: [1, 1]

PASS: Real crops + DINOv2-Small + DFF-Adapter work on the T4.


In [8]:
import subprocess
import sys

CODE_DIR = "/kaggle/input/datasets/saisdsaiiitbh/dff-code"

for script in ["train.py", "evaluate_dff.py", "run_experiments.py"]:
    print(f"\n{'=' * 25} {script} {'=' * 25}")

    result = subprocess.run(
        [sys.executable, f"{CODE_DIR}/{script}", "--help"],
        capture_output=True,
        text=True,
        timeout=30,
    )

    print(result.stdout[:12000])

    if result.returncode != 0:
        print("STDERR:", result.stderr[:3000])
        print("Return code:", result.returncode)


========================= train.py =========================
usage: train.py [-h] [--backbone {tiny,small,base,large}]
                [--crops_root CROPS_ROOT] [--manifest MANIFEST]
                [--out_dir OUT_DIR] [--epochs EPOCHS]
                [--frames_per_video FRAMES_PER_VIDEO]
                [--val_frames VAL_FRAMES] [--batch_size BATCH_SIZE] [--lr LR]
                [--weight_decay WEIGHT_DECAY] [--lambda_auth LAMBDA_AUTH]
                [--lambda_type LAMBDA_TYPE]
                [--label_smoothing LABEL_SMOOTHING] [--no_aux]
                [--variant {dff,lora,linear}] [--num_workers NUM_WORKERS]
                [--max_hours MAX_HOURS] [--ckpt_every CKPT_EVERY]
                [--log_every LOG_EVERY] [--seed SEED] [--smoke]

options:
  -h, --help            show this help message and exit
  --backbone {tiny,small,base,large}
  --crops_root CROPS_ROOT
  --manifest MANIFEST   default: <crops_root>/ffpp_manifest_split.csv
  --out_dir OUT_DIR
  --epochs EPOCHS
  --fram

In [6]:
import os
import subprocess
import sys
from pathlib import Path

CODE_DIR = "/kaggle/input/datasets/saisdsaiiitbh/dff-code"
FF_ROOT = "/kaggle/working/ff-local"
FF_MANIFEST = f"{FF_ROOT}/ffpp_manifest_split.csv"

RUN_DIR = "/kaggle/working/dff_runs/dff_full"
Path(RUN_DIR).mkdir(parents=True, exist_ok=True)

assert Path(f"{CODE_DIR}/train.py").exists(), "train.py not found"
assert Path(FF_MANIFEST).exists(), "FF++ manifest not found"

cmd = [
    sys.executable, "-u", f"{CODE_DIR}/train.py",
    "--backbone", "small",
    "--crops_root", FF_ROOT,
    "--manifest", FF_MANIFEST,
    "--out_dir", RUN_DIR,
    "--epochs", "12",
    "--frames_per_video", "10",
    "--val_frames", "8",
    "--batch_size", "24",
    "--lr", "1e-4",
    "--weight_decay", "0.01",
    "--lambda_auth", "10",
    "--lambda_type", "2",
    "--label_smoothing", "0.0",
    "--variant", "dff",
    "--num_workers", "4",
    "--max_hours", "6",
    "--ckpt_every", "250",
    "--log_every", "50",
    "--seed", "42",
]

print("Starting main DFF-Adapter training")
print("GPU:", __import__("torch").cuda.get_device_name(0))
print("Output:", RUN_DIR)
print("Maximum training time: 4 hours")
print("-" * 60)

result = subprocess.run(cmd, cwd=CODE_DIR)

print("\nTraining process exit code:", result.returncode)
print("\nFiles saved:")
for p in sorted(Path(RUN_DIR).glob("*")):
    print(f"{p.name:25s} {p.stat().st_size / (1024**2):8.1f} MB")

if result.returncode != 0:
    print("\nTraining exited with an error. Inspect the traceback above before continuing.")

Starting main DFF-Adapter training
GPU: Tesla T4
Output: /kaggle/working/dff_runs/dff_full
Maximum training time: 4 hours
------------------------------------------------------------
train: 3600 videos -> 36000 samples/epoch | val: 700 videos, 5600 frames | device=cuda amp=True


Loading weights: 100%|██████████| 223/223 [00:00<00:00, 2333.19it/s]


trainable parameters: 667,374
plan: 12 epochs x 1500 steps = 18000 steps (batch 24, 36000 samples/epoch)
epoch 0: first batch ready after 0.2s
{'epoch': 0, 'step': 50, 'of': 18000, 'loss_auth': 0.6596, 'loss_type': 1.8676, 'acc_type': 0.223, 'samples_s': 96.6, 'eta_min': 74.3}
{'epoch': 0, 'step': 100, 'of': 18000, 'loss_auth': 0.531, 'loss_type': 1.7571, 'acc_type': 0.223, 'samples_s': 103.2, 'eta_min': 69.4}
{'epoch': 0, 'step': 150, 'of': 18000, 'loss_auth': 0.5026, 'loss_type': 1.689, 'acc_type': 0.236, 'samples_s': 102.2, 'eta_min': 69.9}
{'epoch': 0, 'step': 200, 'of': 18000, 'loss_auth': 0.4462, 'loss_type': 1.5995, 'acc_type': 0.279, 'samples_s': 101.6, 'eta_min': 70.1}
{'epoch': 0, 'step': 250, 'of': 18000, 'loss_auth': 0.5011, 'loss_type': 1.5479, 'acc_type': 0.308, 'samples_s': 100.3, 'eta_min': 70.8}
{'epoch': 0, 'step': 300, 'of': 18000, 'loss_auth': 0.4728, 'loss_type': 1.4689, 'acc_type': 0.362, 'samples_s': 98.5, 'eta_min': 71.9}
{'epoch': 0, 'step': 350, 'of': 18000, '

In [10]:
from pathlib import Path
import json

RUN_DIR = Path("/kaggle/working/dff_runs/dff_full")

print("Checkpoint and result files:")
for p in sorted(RUN_DIR.rglob("*")):
    if p.is_file():
        print(f"{p.relative_to(RUN_DIR)}  ({p.stat().st_size / (1024**2):.1f} MB)")

done_file = RUN_DIR / "done.json"
if done_file.exists():
    print("\nCompletion metadata:")
    print(done_file.read_text())
else:
    print("\nNo done.json found; the run may be incomplete.")

print("\nTraining logs, if saved separately:")
for p in RUN_DIR.glob("*.log"):
    print(p.name)

Checkpoint and result files:
args.json  (0.0 MB)
last.pt  (7.8 MB)
log.jsonl  (0.0 MB)

No done.json found; the run may be incomplete.

Training logs, if saved separately:


In [5]:
import subprocess, sys
CODE_DIR = "/kaggle/input/datasets/saisdsaiiitbh/dff-code"
FF_ROOT = "/kaggle/working/ff-local"
FF_MANIFEST = f"{FF_ROOT}/ffpp_manifest_split.csv"

subprocess.run([sys.executable, "-u", f"{CODE_DIR}/bench.py",
                "--crops_root", FF_ROOT, "--manifest", FF_MANIFEST,
                "--num_workers", "4"], cwd=CODE_DIR)

Loading weights: 100%|██████████| 223/223 [00:00<00:00, 2390.51it/s]


data loader alone : 1759.6 samples/s  (workers=4)
GPU step alone    :  108.9 samples/s
=> GPU-BOUND (reduce compute: see options)


CompletedProcess(args=['/usr/bin/python3', '-u', '/kaggle/input/datasets/saisdsaiiitbh/dff-code/bench.py', '--crops_root', '/kaggle/working/ff-local', '--manifest', '/kaggle/working/ff-local/ffpp_manifest_split.csv', '--num_workers', '4'], returncode=0)

In [4]:
import os, shutil, time
from concurrent.futures import ThreadPoolExecutor

SRC = "/kaggle/input/datasets/saisdsaiiitbh/ff-tar"
DST = "/kaggle/working/ff-local"
os.makedirs(DST, exist_ok=True)

files = []
for root, _, fs in os.walk(f"{SRC}/ffpp"):
    for f in fs:
        files.append(os.path.join(root, f))
print("files to copy:", len(files))

def cp(src):
    dst = os.path.join(DST, os.path.relpath(src, SRC))
    if os.path.exists(dst):
        return
    os.makedirs(os.path.dirname(dst), exist_ok=True)
    shutil.copyfile(src, dst)

t = time.time()
with ThreadPoolExecutor(64) as ex:
    list(ex.map(cp, files))
shutil.copyfile(f"{SRC}/ffpp_manifest_split.csv", f"{DST}/ffpp_manifest_split.csv")
print(f"done in {(time.time()-t)/60:.1f} min")
os.system(f"df -h /kaggle/working | tail -1")

files to copy: 165000
done in 2.4 min
/dev/loop2       20G  2.5G   18G  13% /kaggle/working


0

In [7]:
import shutil
from pathlib import Path

source = Path("/kaggle/working/dff_runs/dff_full")
archive_base = "/kaggle/working/dff_full_results"

archive = shutil.make_archive(
    archive_base,
    "zip",
    root_dir=source
)

print("Archive created:", archive)
print("Archive size:", round(Path(archive).stat().st_size / (1024**2), 2), "MB")

Archive created: /kaggle/working/dff_full_results.zip
Archive size: 13.25 MB


In [8]:
import json
from pathlib import Path

run_dir = Path("/kaggle/working/dff_runs/dff_full")

print("Best checkpoint exists:", (run_dir / "best.pt").exists())
print("Completion metadata:")
print((run_dir / "done.json").read_text())

print("\nTraining configuration:")
print((run_dir / "args.json").read_text())

Best checkpoint exists: True
Completion metadata:
{"best_val_auc": 0.990829081632653, "steps": 18000}

Training configuration:
{
  "backbone": "small",
  "crops_root": "/kaggle/working/ff-local",
  "manifest": "/kaggle/working/ff-local/ffpp_manifest_split.csv",
  "out_dir": "/kaggle/working/dff_runs/dff_full",
  "epochs": 12,
  "frames_per_video": 10,
  "val_frames": 8,
  "batch_size": 24,
  "lr": 0.0001,
  "weight_decay": 0.01,
  "lambda_auth": 10.0,
  "lambda_type": 2.0,
  "label_smoothing": 0.0,
  "no_aux": false,
  "variant": "dff",
  "num_workers": 4,
  "max_hours": 6.0,
  "ckpt_every": 250,
  "log_every": 50,
  "seed": 42,
  "smoke": false
}


In [9]:
import subprocess
import sys
from pathlib import Path

CODE_DIR = "/kaggle/input/datasets/saisdsaiiitbh/dff-code"
EVAL_ROOT = "/kaggle/input/datasets/saisdsaiiitbh/eval-tar"
CKPT = "/kaggle/working/dff_runs/dff_full/best.pt"
OUT = "/kaggle/working/dff_runs/dff_full/cross_dataset_eval.json"

cdf1_manifest = f"{EVAL_ROOT}/cdf1_manifest.csv"
cdf2_manifest = f"{EVAL_ROOT}/cdf2_manifest.csv"

assert Path(CKPT).exists(), f"Checkpoint missing: {CKPT}"
assert Path(cdf1_manifest).exists(), f"Manifest missing: {cdf1_manifest}"
assert Path(cdf2_manifest).exists(), f"Manifest missing: {cdf2_manifest}"

cmd = [
    sys.executable, "-u", f"{CODE_DIR}/evaluate_dff.py",
    "--ckpt", CKPT,
    "--backbone", "small",
    "--variant", "dff",
    "--sets",
    f"{cdf1_manifest}:all",
    f"{cdf2_manifest}:all",
    "--frames", "0",
    "--batch_size", "32",
    "--num_workers", "2",
    "--out", OUT,
]

print("Evaluating best checkpoint on CDF-v1 and CDF-v2...")
result = subprocess.run(cmd, cwd=CODE_DIR)
print("Exit code:", result.returncode)

if result.returncode == 0:
    print("\nEvaluation output:")
    if Path(OUT).exists():
        print(Path(OUT).read_text())
else:
    print("Evaluation failed. Inspect the traceback above before retrying.")

Evaluating best checkpoint on CDF-v1 and CDF-v2...


Loading weights: 100%|██████████| 223/223 [00:00<00:00, 2333.34it/s]


/kaggle/input/datasets/saisdsaiiitbh/eval-tar/cdf1_manifest.csv:all AUC=81.20  EER=30.65  frameAUC=77.47  (100 videos, 3200 frames)
/kaggle/input/datasets/saisdsaiiitbh/eval-tar/cdf2_manifest.csv:all AUC=85.02  EER=22.94  frameAUC=77.71  (518 videos, 16576 frames)
Exit code: 0

Evaluation output:
{
  "/kaggle/input/datasets/saisdsaiiitbh/eval-tar/cdf1_manifest.csv:all": {
    "video_auc": 0.8119694397283531,
    "video_eer": 0.30645161290322576,
    "frame_auc": 0.774682244137309,
    "n_videos": 100,
    "n_frames": 3200
  },
  "/kaggle/input/datasets/saisdsaiiitbh/eval-tar/cdf2_manifest.csv:all": {
    "video_auc": 0.8502313284864508,
    "video_eer": 0.22941176470588232,
    "frame_auc": 0.7770855466813656,
    "n_videos": 518,
    "n_frames": 16576
  }
}


In [10]:
# Backuop

import os
import shutil

source = "/kaggle/working/dff_runs/dff_full"
backup = "/kaggle/working/dff_baseline_backup.zip"

assert os.path.exists(os.path.join(source, "best.pt")), "Baseline checkpoint missing!"

shutil.make_archive(
    "/kaggle/working/dff_baseline_backup",
    "zip",
    root_dir=source
)

print(f"Backup created: {backup}")
print(f"Size: {os.path.getsize(backup) / (1024**2):.2f} MB")

Backup created: /kaggle/working/dff_baseline_backup.zip
Size: 13.25 MB


### Design

# DFF-Adapter reproduction: design note (Stage 0) and deviation log

Paper: arXiv:2511.12107 (Zhang et al.). Backbone: frozen DINOv2. Status key:
**P** = stated in the paper, **A** = our assumption (paper ambiguous), **D** = deliberate deviation.

## 1. One-paragraph summary (say this first in the viva)
A DINOv2 block has query/key/value linears, an attention output linear, and an MLP. We freeze all of
DINOv2 and wrap three linears per block (query, value, attention-output dense) with a small trainable
adapter. The adapter splits the layer's input channels into 4 chunks. Chunk 0 goes to a *shared* head
(softmax mix of all 6 experts); chunks 1-3 go to *task* heads (each mixes its top-3 of 6 experts).
An expert is a LoRA pair (down-project to rank 4, up-project back). The same network is run twice per
batch: task 0 (real/fake, BCE loss) and task 1 (forgery type, cross-entropy). The two tasks share the
experts and the shared head, but use different routing logits for the task heads, so what the
forgery-type task learns reaches the authenticity task through the shared weights.
Only the adapters and two linear heads train. At test time only task 0 is used.

## 2. Shapes (DINOv2-Small: d=384, 12 blocks, 6 attention heads; 224x224 -> 256 patches + CLS)
| tensor | shape |
|---|---|
| token stream in every block | (B, 257, 384) |
| query / value / output-dense input and output | (B, 257, 384) |
| adapter chunk | (B, 257, 4, 96) |
| expert A (down) / B (up) per head | (6, 4, 96) / (6, 96, 4) |
| routing logits Z_task / Z_shared per adapter | (2, 3, 6) / (6,) |
| CLS token to heads | (B, 384) |

## 3. Equation -> code map (`dff_adapter.py`)
| Paper | Meaning | Code |
|---|---|---|
| Eq. 4 | g = softmax(Z_task[t,k]) | `gate_weights`: `F.softmax(self.z_task[t])` |
| Eq. 3 | S = Top3(N, g) | `p.topk(self.cfg.top_k)` |
| Eq. 2 | renormalise over the selected 3 | `top_v / top_v.sum(...)` then `scatter` |
| Eq. 1 | f = beta * sum_j g~_j B_j(A_j x) | `forward`: two `einsum`s, times `self.scale` |
| Eq. 5 | shared head: softmax over all N experts | row 0 of `gate_weights` |
| Eq. 6, 8 | concat(shared, task heads), added residually | head outputs concatenated to width d, added to the frozen linear's output in `DFFLinear.forward` |
| Eq. 7 | L_bce on authenticity head | `train.py` (to be written) |
| Eq. 9 | L_ftc on forgery-type head | `train.py` |
| Eq. 10 | L = l0 L_bce + l1 L_ftc | `train.py` |
| "inserted into every query, value, dense" | injection | `inject_dff` |
| "task flag 0/1, two forward passes" | task switching | `TaskContext.task`, `DFFDetector.forward(task=...)` |

`test_matches_naive_equations` checks the batched code against a literal loop implementation of Eq. 1-5.

## 4. Decisions
| # | Decision | Status | Evidence / reasoning |
|---|---|---|---|
| D1 | Router logits are learned tensors, not computed from the input | P | Eq. 4-5 define Z as parameters. The text says the router "scores adapters that fit the statistics of the sub-space", which suggests input-dependence, but the equations do not. We follow the equations. |
| D2 | H=4 = 1 shared head (chunk 0) + 3 task heads (chunks 1-3) | A | Paper: "4 head adapters"; Eq. 6: concat(f_share, f^1..f^n); X^(0) is the shared head's slice. Concat width = d only if 1 + n = 4. |
| D3 | Each head has its own bank of N=6 experts | A | Table 9: "experts per head". Gives 3.54M adapter params for DINOv2-Large vs paper's 3.5M. One bank shared by all heads would give 0.89M. Supporting evidence, not proof. |
| D4 | Rank per head = r/H = 4; scale beta = alpha/r = 2 | P/A | Paper gives r_h = r/h. Scale = alpha/r is the standard LoRA convention; the paper just says "scaling coefficient". |
| D5 | Fusion is the per-layer residual addition; one CLS token, two forward passes | A | Paper's training section describes two passes with a task flag. Fig. 2 draws two CLS tokens; we read that as one token position evaluated under two task flags. |
| D6 | "dense" = attention output projection | A | Paper says "query, value, and dense projection of each block"; Fig. 2 shows q, v and MLP. We did not adapt MLP layers. |
| D7 | Routing logits initialised N(0, 0.5^2); B zero-init, A uniform(+-1/sqrt(d_h)) | A | Paper silent. Zero B makes the adapter an exact no-op at init (tested). |
| D8 | 5 forgery-type classes: real + Deepfakes, Face2Face, FaceSwap, NeuralTextures | A | From Fig. 1 (Real, FaceSwap, Face2Face, ...). |

## 5. Known behaviour worth knowing before the viva
- **Dead experts.** Top-k routing gives zero gradient to experts outside the selected 3 (the softmax
  Jacobian cancels after renormalising). An expert unselected by both tasks in a given head/layer never
  trains. `routing_report` counts these. This also affects the paper's own formulation (it is a property
  of top-k with static logits), so we report it as an observation, not fix it silently.
- **Why query/value/dense, not key?** The paper does not justify it. LoRA literature commonly adapts
  q and v; do not claim the paper proves this.

## 6. Deviation log (fill in as we go)
| Item | Paper | Ours | Reason | Expected effect |
|---|---|---|---|---|
| Backbone | DINOv2-Large with registers | Small or Base (no registers) | Free-tier GPU budget | Lower absolute AUC; compare with the paper's Table 6 Base rows |
| Face preprocessing | DeepfakeBench pipeline | MTCNN, 1.3x square crop, 224, 32 frames/video | Tooling available; 0.02% frames had no detection | Small domain shift |
| Frames per epoch | 32 per video, 50 epochs | sampled subset, few epochs | GPU budget | Lower AUC |
| Batch size / lr / lambdas | 24 / 2e-4 / 10, 2 | TBD | Memory | TBD |
| Cross-dataset sets | CDF-v1, CDF-v2, DFDC, DFDCP | CDF-v1 (100 test videos), CDF-v2 (518) | DFDCP access unavailable | No DFDCP column; CDF-v1 AUC is noisy |
| Splits | DeepfakeBench / official | Official FF++ 720/140/140 | Same | None |

## 7. Problems found in the Phase-2 submission (fixed in this rebuild)
1. One expert bank, router and shared vector shared across all 36 wrapped layers (about 14k adapter
   parameters vs about 0.67M here).
2. Fusion used input chunk 0 twice and never used chunk 3.
3. Hyperparameters: lambda 1/1, lr 1e-4, batch 2 (paper: 10/2, 2e-4, 24).
4. Custom 400/50/50 random pair split instead of the official 360/70/70 pairs.